In [14]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from interpret import show
from interpret.glassbox import ClassificationTree
from interpret.glassbox import LogisticRegression
from sklearn.decomposition import PCA
from interpret.blackbox import ShapKernel
from sklearn.pipeline import Pipeline
from interpret.blackbox import LimeTabular
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

'''
Data: breast cancer and adult data
Glassbox model: logistic regression
Blackbox model: random forest
Explenation models: shap and lime

Goal:
1-Take the breast cancer data and make a logistic regression model from it
2-Explain the logistic regression model with both shap and lime
3-Repeat steps 1-2 using a random forest model instead
4-Repeat steps 1-3 using the adult data instead
'''


def load_breast_data():
    breast = load_breast_cancer()
    feature_names = list(breast.feature_names)
    X, y = pd.DataFrame(breast.data, columns=feature_names), breast.target
    datasetbc = {
        'problem': 'classification',
        'full': {
            'X': X,
            'y': y,
        },
    }
    return datasetbc


def load_adult_data():
    df = pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        header=None)
    df.columns = [
        "Age", "WorkClass", "fnlwgt", "Education", "EducationNum",
        "MaritalStatus", "Occupation", "Relationship", "Race", "Gender",
        "CapitalGain", "CapitalLoss", "HoursPerWeek", "NativeCountry", "Income"
    ]
    train_cols = df.columns[0:-1]
    label = df.columns[-1]
    X_df = df[train_cols]
    y_df = df[label]

    # Preprocess categorical variables
    categorical_cols = ['WorkClass', 'Education', 'MaritalStatus', 'Occupation', 'Relationship', 'Race', 'Gender', 'NativeCountry']
    X_df = pd.get_dummies(X_df, columns=categorical_cols, drop_first=True)
    
    # Encode target
    le = LabelEncoder()
    y_df = le.fit_transform(y_df)

    datasetad = {
        'problem': 'classification',
        'full': {
            'X': X_df,
            'y': y_df,
        },
    }

    return datasetad

In [15]:
# Use a logistic regression to analyze the breast cancer data
datasetbc = load_breast_data()
X_train, X_test, y_train, y_test = train_test_split(datasetbc['full']['X'], datasetbc['full']['y'], test_size=0.20, random_state=42)

pca = PCA()
lr = LogisticRegression(random_state=42)

glassbox_modelbc = Pipeline([('pca', pca), ('lr', lr)])
glassbox_modelbc.fit(X_train, y_train)

y_pred_proba = glassbox_modelbc.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")

AUC Score: 0.9980


In [16]:
# Display some random entries from the breast cancer dataset
print('Random entries from breast cancer dataset:')
datasetbc['full']['X'].sample(5)

Random entries from breast cancer dataset:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
498,18.490,17.52,121.30,1068.0,0.10120,0.13170,0.14910,0.091830,0.1832,0.06697,...,22.75,22.88,146.40,1600.0,0.14120,0.3089,0.35330,0.16630,0.2510,0.09445
467,9.668,18.10,61.06,286.3,0.08311,0.05428,0.01479,0.005769,0.1680,0.06412,...,11.15,24.62,71.11,380.2,0.13880,0.1255,0.06409,0.02500,0.3057,0.07875
32,17.020,23.98,112.80,899.3,0.11970,0.14960,0.24170,0.120300,0.2248,0.06382,...,20.88,32.09,136.10,1344.0,0.16340,0.3559,0.55880,0.18470,0.3530,0.08482
450,11.870,21.54,76.83,432.0,0.06613,0.10640,0.08777,0.023860,0.1349,0.06612,...,12.79,28.18,83.51,507.2,0.09457,0.3399,0.32180,0.08750,0.2305,0.09952
169,14.970,16.95,96.22,685.9,0.09855,0.07885,0.02602,0.037810,0.1780,0.05650,...,16.11,23.00,104.60,793.7,0.12160,0.1637,0.06648,0.08485,0.2404,0.06428


In [17]:
# use shap to explain the logistic regression model on breast cancer data
import shap as shap_module
shap_explainer = ShapKernel(glassbox_modelbc, shap_module.sample(X_train, 100))
shap_local = shap_explainer.explain_local(X_test[:5], y_test[:5])
show(shap_local, 0)

  0%|          | 0/5 [00:00<?, ?it/s]

<!-- http://127.0.0.1:7001/3081985078608/ -->

In [18]:
# use lime to explain the decision list model on breast cancer data
lime = LimeTabular(glassbox_modelbc, X_train)
lime_local = lime.explain_local(X_test[:5], y_test[:5])
show(lime_local, 0)

<!-- http://127.0.0.1:7001/3081985077840/ -->

In [19]:
# use random forest on the breast cancer dataset and show the results
rf = RandomForestClassifier(random_state=42)
blackbox_modelbc = Pipeline([('pca', pca), ('rf', rf)])
blackbox_modelbc.fit(X_train, y_train)
y_pred_proba = blackbox_modelbc.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")

AUC Score: 0.9910


In [20]:
# use shap to explain the random forest model on breast cancer data
import shap as shap_module
shap_explainer = ShapKernel(blackbox_modelbc, shap_module.sample(X_train, 100))
shap_local = shap_explainer.explain_local(X_test[:5], y_test[:5])
show(shap_local, 0)

  0%|          | 0/5 [00:00<?, ?it/s]

<!-- http://127.0.0.1:7001/3082251930272/ -->

In [21]:
# use lime to explain the random forest model on breast cancer data
lime = LimeTabular(blackbox_modelbc, X_train)
lime_local = lime.explain_local(X_test[:5], y_test[:5])
show(lime_local, 0)

<!-- http://127.0.0.1:7001/3082274867632/ -->

In [22]:
# Use a logistic regression to analyze the adult data
dataseta = load_adult_data()
X_train, X_test, y_train, y_test = train_test_split(dataseta['full']['X'], dataseta['full']['y'], test_size=0.20, random_state=42)

pca = PCA()
lr = LogisticRegression(random_state=42)

glassbox_modela = Pipeline([('pca', pca), ('lr', lr)])
glassbox_modela.fit(X_train, y_train)

y_pred_proba = glassbox_modela.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")

AUC Score: 0.8214


In [23]:
# use shap to explain the logistic regression model on adult data
import shap as shap_module
shap_explainer = ShapKernel(glassbox_modela, shap_module.sample(X_train, 100))
shap_local = shap_explainer.explain_local(X_test[:5], y_test[:5])
show(shap_local, 0)

  0%|          | 0/5 [00:00<?, ?it/s]

<!-- http://127.0.0.1:7001/3082274676464/ -->

In [24]:
# use lime to explain the decision list model on adult data
lime = LimeTabular(glassbox_modela, X_train)
lime_local = lime.explain_local(X_test[:5], y_test[:5])
show(lime_local, 0)

<!-- http://127.0.0.1:7001/3082251950512/ -->

In [25]:
# use random forest on the adult dataset and show the results
rf = RandomForestClassifier(random_state=42)
blackbox_modela = Pipeline([('pca', pca), ('rf', rf)])
blackbox_modela.fit(X_train, y_train)
y_pred_proba = blackbox_modela.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")

AUC Score: 0.8983


In [26]:
# use shap to explain the random forest model on adult data
import shap as shap_module
shap_explainer = ShapKernel(blackbox_modela, shap_module.sample(X_train, 100))
shap_local = shap_explainer.explain_local(X_test[:5], y_test[:5])
show(shap_local, 0)

  0%|          | 0/5 [00:00<?, ?it/s]

<!-- http://127.0.0.1:7001/3082052339408/ -->

In [27]:
# use lime to explain the random forest model on adult data
lime = LimeTabular(blackbox_modela, X_train)
lime_local = lime.explain_local(X_test[:5], y_test[:5])
show(lime_local, 0)

<!-- http://127.0.0.1:7001/3082252417168/ -->

In [28]:
# Display some random entries from the adult dataset
print('Random entries from adult dataset:')
dataseta['full']['X'].sample(5)

Random entries from adult dataset:


,Age,fnlwgt,EducationNum,CapitalGain,CapitalLoss,HoursPerWeek,WorkClass_ Federal-gov,WorkClass_ Local-gov,WorkClass_ Never-worked,WorkClass_ Private,...,NativeCountry_ Portugal,NativeCountry_ Puerto-Rico,NativeCountry_ Scotland,NativeCountry_ South,NativeCountry_ Taiwan,NativeCountry_ Thailand,NativeCountry_ Trinadad&Tobago,NativeCountry_ United-States,NativeCountry_ Vietnam,NativeCountry_ Yugoslavia
20392,33,219034,7,0,0,40,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
2210,39,179352,10,0,0,35,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
17535,64,631947,6,0,0,40,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
22059,29,26451,9,0,0,40,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
31321,59,184493,10,0,0,40,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
